# AF2·10 — Dissect Real AlphaFold2 (Capstone)

**The victory lap.** You have built every mechanism in AlphaFold2 from scratch and wired
them into a toy that folds. Now meet the real thing: run **ColabFold** (production
AlphaFold2) on a real protein sequence, and read its outputs — pLDDT, PAE, the recycles,
the structure — recognising each as something you built.

Nothing here is new machinery. It is the same coevolution signal (rung 01), the same
Evoformer trunk (02–04), the same frames and IPA and FAPE (05–07), the same recycling and
confidence heads (08), the same end-to-end trunk→structure flow (09) — just at full scale,
with real weights and a real MSA. The goal of this notebook is *recognition*: to look at a
real AlphaFold output and see your own notebooks in it.

> **Compute note.** Running real AlphaFold2 needs a **GPU** and an internet MSA search — a
> free Google **Colab GPU runtime** is the standard way. The ColabFold cell below is
> written for that environment and is **guarded off by default** (`RUN_COLABFOLD = False`)
> so the rest of the notebook runs anywhere. The analysis reps operate on a clearly-labelled
> **synthetic placeholder** output so you can build and test them offline, then point them at
> your real ColabFold results by flipping one flag.

**How to use this notebook:** implement the analysis reps and make the checkpoints pass
(they run on the placeholder, CPU-only). Then, on Colab with a GPU, set
`RUN_COLABFOLD = True`, fold a real sequence, and rerun the analysis on the real arrays.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK, RED = '#2a78d6', '#008300', '#52514e', '#e34948'

RUN_COLABFOLD = False        # <- set True ONLY on a Colab GPU runtime (see the next cell)
print('RUN_COLABFOLD =', RUN_COLABFOLD, '(analysis below uses a synthetic placeholder unless True)')

## Part 1 — running the real AlphaFold2 (Colab GPU)

On a Colab GPU runtime, ColabFold installs in a couple of minutes and folds a sequence with
a few lines. The cell below is the standard invocation. It:

1. **searches for homologs** to build an MSA (the MMseqs2 server) — this is rung 01's MSA,
   fetched for real instead of synthesized;
2. runs the **Evoformer trunk + structure module** with real AlphaFold2 weights (rungs 02–07)
   across several **recycles** (rung 08);
3. writes a **PDB** structure plus per-residue **pLDDT** and a **PAE** matrix (rung 08's
   confidence heads).

It is guarded by `RUN_COLABFOLD` so it does not attempt to run off-GPU. (This cell was
authored but **not executed** in a CPU environment — it is meant for Colab.)

In [ ]:
# --- Colab GPU only ---  set RUN_COLABFOLD = True above to use.
if RUN_COLABFOLD:
    # one-time install (Colab):
    #   !pip -q install "colabfold[alphafold]"
    from colabfold.batch import run
    sequence = "GSHMSLEEKKGNYVVLDKETGKVTGEAIKQAIQNIDPTKPFVLAVKSGKEIFEKPLTAEEAEEFKKAFEE"  # your sequence
    results = run(
        queries=[("my_protein", sequence, None)],
        result_dir="af_out",
        num_models=1, num_recycles=3, use_templates=False,
        msa_mode="mmseqs2_uniref_env",
    )
    # ColabFold writes: my_protein_..._scores.json  (contains 'plddt' and 'pae')
    import json, glob
    scores = json.load(open(sorted(glob.glob("af_out/*_scores*.json"))[0]))
    plddt = np.array(scores["plddt"])          # [L] per-residue confidence, 0..100
    pae   = np.array(scores["pae"])            # [L, L] predicted aligned error, angstrom
    print("folded! length", len(plddt), "mean pLDDT %.1f" % plddt.mean())
else:
    print("skipped (set RUN_COLABFOLD = True on a Colab GPU to fold a real sequence)")

## Part 2 — a stand-in output, so the analysis runs anywhere

So that the analysis cells run without a GPU, here is a **synthetic placeholder** that mimics
the shape of a real ColabFold result: a two-domain protein joined by a flexible linker —
exactly the case where pLDDT and PAE earn their keep. **This is fabricated for offline
testing**; on Colab, the previous cell overwrites `plddt` and `pae` with the real arrays and
every cell below works unchanged.

In [ ]:
if not RUN_COLABFOLD:
    rng = np.random.default_rng(0)
    L = 90
    dom1 = np.arange(0, 40); linker = np.arange(40, 50); dom2 = np.arange(50, 90)
    # pLDDT: high inside folded domains, low in the disordered linker (as real AF2 shows)
    plddt = np.full(L, 90.0)
    plddt[linker] = 45.0
    plddt += rng.normal(0, 4, L); plddt = np.clip(plddt, 20, 98)
    # PAE: low WITHIN each domain (confidently packed), high BETWEEN domains
    # (their relative orientation is uncertain — the two domains float on the linker)
    pae = np.full((L, L), 3.0)
    for dom in (dom1, dom2):
        pae[np.ix_(dom, dom)] = 1.5 + rng.normal(0, 0.3, (len(dom), len(dom))).clip(-1, 3)
    inter = 22.0
    pae[np.ix_(dom1, dom2)] = inter; pae[np.ix_(dom2, dom1)] = inter
    pae[:, linker] = 18.0; pae[linker, :] = 18.0
    pae = np.clip(pae, 0.3, 31.0); pae = 0.5 * (pae + pae.T)
    print("synthetic placeholder: L=%d, two domains + a linker" % L)
print("plddt shape", plddt.shape, "| pae shape", pae.shape)

## Part 3 — pLDDT: which residues to trust

pLDDT is the **per-residue confidence** you built in rung 08 — AlphaFold's estimate of how
accurately each residue is placed, on a 0–100 scale (higher is better). In practice: `>90`
very high, `70–90` confident, `50–70` low, `<50` usually disordered/flexible. The first thing
anyone does with an AlphaFold model is colour it by pLDDT and read off which parts to believe.

### Rep 1 — `confident_fraction(plddt, thresh=70)`
Return the fraction of residues with `pLDDT >= thresh`. A quick global quality read.

In [ ]:
def confident_fraction(plddt, thresh=70):
    '''Fraction of residues at or above a pLDDT threshold.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
cf = confident_fraction(plddt)
assert 0.0 <= cf <= 1.0
# the folded domains are confident; the linker is not
assert confident_fraction(plddt) > confident_fraction(plddt, thresh=95), 'fewer pass a stricter cutoff'
assert plddt.min() < 70 < plddt.max(), 'this example has both confident and unconfident residues'
print('%.0f%% of residues are confident (pLDDT >= 70)' % (100 * cf))

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.fill_between(np.arange(len(plddt)), plddt, color=BLUE, alpha=.3)
ax.plot(plddt, color=BLUE, lw=1.5)
ax.axhline(70, color=INK, ls='--', lw=1); ax.text(0, 71, 'confident', fontsize=8, color=INK)
ax.set_xlabel('residue'); ax.set_ylabel('pLDDT'); ax.set_ylim(0, 100)
ax.set_title('per-residue confidence — the dip is a flexible linker'); ax.grid(alpha=.15)
plt.show()

## Part 4 — PAE: are the domains placed relative to each other?

PAE (predicted aligned error) is the **per-pair confidence** from rung 08: `PAE[i,j]` is the
expected error in residue `j`'s position when the structure is aligned on residue `i`. Its
real power is **multi-domain analysis**. A protein can have two domains that are each folded
with high pLDDT, yet float uncertainly relative to each other on a flexible linker. pLDDT
cannot see this (both domains look confident); PAE shows it immediately — **low PAE within
each domain, high PAE between them**. Reading a PAE plot is how you tell "one rigid body"
from "several domains on strings".

### Rep 2 — `mean_pae_between(pae, block_a, block_b)`
Return the mean PAE between two residue-index groups, `pae[block_a][:, block_b].mean()` —
the confidence in how `block_b` is positioned relative to `block_a`.

In [ ]:
def mean_pae_between(pae, block_a, block_b):
    '''Mean predicted aligned error between two groups of residues.'''
    # YOUR CODE HERE
    # hint: pae[np.ix_(block_a, block_b)].mean()
    raise NotImplementedError

# --- checkpoint ---
d1 = np.arange(0, 40); d2 = np.arange(50, 90)
within = mean_pae_between(pae, d1, d1)
between = mean_pae_between(pae, d1, d2)
assert between > within + 3, 'inter-domain PAE should be much higher than intra-domain'
print('mean PAE within domain 1: %.1f A   |   between domain 1 and 2: %.1f A' % (within, between))
print('-> each domain is internally confident, but their relative placement is not.')

### Rep 3 — `worst_positioned(pae)`
Return the index of the residue with the highest **mean** PAE over all partners — the residue
whose placement relative to the rest of the structure is least certain (typically deep in a
flexible/linker region). A one-liner that turns the `[L,L]` PAE into a per-residue red flag.

In [ ]:
def worst_positioned(pae):
    '''Index of the residue with the highest average PAE to all others.'''
    # YOUR CODE HERE
    # hint: pae.mean(axis=1).argmax()
    raise NotImplementedError

# --- checkpoint ---
w = worst_positioned(pae)
assert 0 <= w < len(plddt)
# it should land in a low-confidence region (the linker or a floating domain edge)
assert plddt[w] < np.median(plddt), 'the worst-positioned residue should not be a confident one'
print('least confidently positioned residue: %d (pLDDT there = %.0f)' % (w, plddt[w]))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
im = axes[0].imshow(pae, cmap='Greens_r', vmin=0, vmax=31)
axes[0].set_title('PAE[i, j] (real AF2 shows exactly this shape)', fontsize=10)
axes[0].set_xlabel('residue j'); axes[0].set_ylabel('residue i (aligned on)')
fig.colorbar(im, ax=axes[0], fraction=0.046, label='expected error (A)')
axes[1].plot(pae.mean(1), color=GREEN, lw=1.5); axes[1].axvline(w, color=RED, ls='--', lw=1)
axes[1].set_xlabel('residue'); axes[1].set_ylabel('mean PAE to all others')
axes[1].set_title('per-residue positional uncertainty', fontsize=10); axes[1].grid(alpha=.15)
plt.tight_layout(); plt.show()
print('The dark off-diagonal blocks = "these two domains are not confidently placed relative')
print('to each other". That single glance is why PAE is in every AlphaFold figure. ✓')

## Part 5 — the map: every real-AF2 output ↔ a rung you built

| Real AlphaFold2 component | What it is | You built it in |
|---|---|---|
| MSA input | homologs found by MMseqs2 | **01** (coevolution), **02** (reading the MSA) |
| Evoformer trunk | 48 blocks of axial + triangle attention | **02** axial attn, **03** triangle ops, **04** the block |
| Distogram head | binned `Cα–Cα` distance distribution | **04** distogram head |
| Structure module | 8 shared IPA layers editing residue frames | **05** frames, **06** IPA, **07** frame updates |
| FAPE | the training loss on the structure | **07** FAPE |
| Recycling | 3+ passes feeding predictions back | **08** recycling |
| pLDDT | per-residue confidence (the b-factor colour) | **08** pLDDT head |
| PAE | per-pair aligned error (the domain plot) | **08** PAE head |
| MSA → structure | the whole forward pass | **09** end-to-end |

When you open an AlphaFold model in a viewer and colour it by pLDDT, that colour is rung 08.
When you look at the PAE plot to decide whether two domains are rigid, that plot is rung 08.
When it folds the sequence at all, that is rungs 01–07 working together as in rung 09. There
is no part of the picture you have not built.

## Reflection — zero to hero, complete

You started not knowing AlphaFold's internals and ended having **built every one of them**
and wired them into a working (toy) folder — then read the real model's output fluently,
because you recognise every number in it.

The through-line of the whole series:

- **Structure hides in sequences** (coevolution), and reading it well requires **global,
  geometry-aware reasoning** — triangle consistency on a pair graph — not pairwise statistics.
- **Geometry belongs in frames, not coordinates.** Residue frames, invariant point attention,
  and FAPE all encode 3D structure in a way that respects the symmetry of space *by
  construction* — the recurring theme that architecture beats hoping a network learns a
  symmetry.
- **A good model knows what it doesn't know.** pLDDT and PAE are what make AlphaFold usable in
  practice, and they are simple heads trained to predict the model's own error.

**Where to go next.** The real system scales every piece here (more blocks, deeper MSAs, the
full all-atom structure module with sidechain torsions, templates, self-distillation) — all
extensions of mechanisms you now own. If you want to keep going: read the AlphaFold2 methods
supplement (it will feel like annotated versions of these notebooks), or look at AlphaFold-
Multimer, ESMFold (a language-model front-end instead of an MSA), and the diffusion-based
successors (AlphaFold3, RFdiffusion) — several of which connect straight back to the
generative-cores track of this very repo.

You didn't learn AlphaFold by running someone's weights. You built it. That transfers.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def confident_fraction(plddt, thresh=70):
    return float((np.asarray(plddt) >= thresh).mean())

def mean_pae_between(pae, block_a, block_b):
    return float(np.asarray(pae)[np.ix_(np.asarray(block_a), np.asarray(block_b))].mean())

def worst_positioned(pae):
    return int(np.asarray(pae).mean(axis=1).argmax())

print('reference solutions loaded — re-run the checkpoint cells above')